<a href="https://colab.research.google.com/github/GuanghuaShi/Tennis-Shot-Recognition/blob/main/Skeleton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Load split

In [1]:
!git clone https://github.com/GuanghuaShi/Tennis-Shot-Recognition.git

Cloning into 'Tennis-Shot-Recognition'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 19 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 47.70 KiB | 15.90 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [2]:
import sys
sys.path.insert(0, "/content/Tennis-Shot-Recognition")

In [4]:
import json

with open("data_splits.json", "r") as f:
    splits = json.load(f)

train_data = splits["train"]
val_data   = splits["val"]
test_data  = splits["test"]

print(len(train_data), len(val_data), len(test_data))

259 56 56


### 2. Setup

In [5]:
import numpy as np
import cv2
from MoveNet import run_inference

HEIGHT, WIDTH = 135, 240
NUM_FRAMES = 32

frame_numbers = [2, 4, 6, 8, 10, 12, 14, 16, 17, 19, 20, 22, 23, 25, 26, 28,
                 29, 31, 32, 34, 35, 37, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56]

Loading MoveNet model...
MoveNet loaded.


### 3. Process ONE video

In [6]:
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)

    frames = []

    for frame_number in frame_numbers:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number - 1)
        ret, frame = cap.read()

        if not ret:
            frame = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

        # 🔥 skeleton extraction
        frame = run_inference(HEIGHT, WIDTH, frame)

        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)

    cap.release()

    return np.array(frames)  # (32, H, W, 3)

### 4. Build dataset from split

In [7]:
def build_dataset(data):
    X, y = [], []

    for video_path, label in data:
        print(video_path)

        frames = process_video(video_path)

        X.append(frames)
        y.append(label)

    return np.array(X), np.array(y)

### Generate datasets

In [8]:
X_train, y_train = build_dataset(train_data)
X_val, y_val     = build_dataset(val_data)
X_test, y_test   = build_dataset(test_data)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/smash/smash12.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/serve/serve35.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/forehand/forehand34.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/b_volley/b_volley44.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/f_volley/f_volley3.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/smash/smash15.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/serve/serve7.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/b_slice/b_slice6.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/forehand/forehand48.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/backhand/backhand18.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/serve/serve17.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/smash/smash48.mp4
/content/3DCNN-Tennis-Action-Recognition/tennis_dataset/backhand/backhand43.mp4
/

### Save

In [9]:
np.save('skeleton32_train_frames.npy', X_train)
np.save('skeleton32_train_classes.npy', y_train)

np.save('skeleton32_val_frames.npy', X_val)
np.save('skeleton32_val_classes.npy', y_val)

np.save('skeleton32_test_frames.npy', X_test)
np.save('skeleton32_test_classes.npy', y_test)